# Jade
### Stockfish con decisiones más variadas y frecuencias de partidas humanas

**Versión 1.2 · Reloj, respuestas pausadas, candidatas humanas y análisis de PGN.**

**Si vienes de la versión anterior:** ejecuta primero la celda 7 en el cuaderno antiguo
y espera a que confirme «Guardado en Drive». Conserva ese cuaderno y la carpeta `Jade`.
Abre este archivo como otro cuaderno, usa la misma cuenta y el mismo `RITMO`, y ejecuta
1, 2, 3 y 4. Comprueba que reaparece tu total de partidas antes de continuar con 6.
No necesitas ejecutar 5 otra vez. Usa un único cuaderno activo por memoria.
El formato de la memoria, los identificadores y los cursores siguen siendo los mismos.
Las partidas guardadas por la interfaz 1.1 también se recuperan. Como no tenían reloj,
continúan sin reloj; empieza una partida nueva para activar el control de tiempo.
Si usas un tablero anterior que no guardaba partidas, exporta su PGN antes de cerrar
o aplica el actualizador en esa misma sesión.

Este cuaderno incluye todo el código. **Solo necesitas subir este `.ipynb` a Google Colab.**
No hace falta subir archivos `.py`, descargar el dataset ni crear cuentas de Hugging Face.

**Primera vez:** usa un entorno Python 3 con acelerador **Ninguno / CPU** y ejecuta
las celdas 1 a 6, en orden. La celda 2 pedirá conectar Drive si está activado.
La celda 5 incorpora hasta 2.000 partidas nuevas; después aparece el tablero en la 6.
Para probar inmediatamente puedes omitir la 5: Jade juega con Stockfish y temperatura.

**Otro día:** ejecuta 1 a 4 para recuperar lo aprendido y 6 para jugar. Ejecuta 5
solo si quieres añadir otro lote. La celda 7 permite guardar manualmente.

No se necesita GPU: esta versión combina búsqueda Stockfish con un número variable
de candidatas, Boltzmann y frecuencias por posición. Los perfiles 1200–1600 son experimentales,
no ratings medidos. El rating de Lichess es Glicko-2 y no equivale directamente a FIDE.

Colab es temporal. Su FAQ restringe el entrenamiento de ajedrez en entornos gratuitos
sin saldo positivo de unidades de computación. Consulta las condiciones de tu plan.


## 1 · Preparar Colab
Instala bibliotecas y el binario de Stockfish de Ubuntu. La versión del motor
depende de la imagen de Colab; se muestra y registra al iniciar Jade.
El paquete incluye la red NNUE necesaria, por lo que no debes buscarla aparte.
No se usa un servidor público: el tablero vive dentro de este cuaderno.


In [ ]:
#@title 1. Instalar dependencias
%pip -q install "python-chess==1.999" "numpy>=1.26,<3" "pyarrow>=17,<26" "huggingface_hub>=0.28,<2" "ipywidgets>=8.1,<9" fsspec

import shutil
import subprocess
from pathlib import Path

if not (shutil.which("stockfish") or Path("/usr/games/stockfish").is_file()):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "--no-install-recommends", "stockfish"], check=True)

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
print("Instalación preparada.")


## 2 · Elegir datos y almacenamiento
Deja los valores iniciales para comenzar. `rapid` usa partidas rápidas; `blitz`
crea otra memoria independiente. No se mezclan ratings de ritmos distintos.

**Drive guarda copias de lo aprendido, no los Parquet.** La carpeta se crea
automáticamente en `Mi unidad/Jade`. Si desactivas Drive, solo habrá copias
temporales en Colab y deberás descargarlas antes de que termine la sesión.
Usa un solo cuaderno activo por memoria para no crear ramas de aprendizaje separadas.


In [ ]:
#@title 2. Ajustes de Jade
GUARDAR_EN_DRIVE = True #@param {type:"boolean"}
RITMO = "rapid" #@param ["rapid", "blitz"]
ANO = 2024 #@param {type:"integer"}
MES = 1 #@param {type:"integer"}
NUEVAS_PARTIDAS = 2000 #@param {type:"integer"}
MAX_FILAS = 50000 #@param {type:"integer"}
MAX_ARCHIVOS = 3 #@param {type:"integer"}
NODOS_STOCKFISH = 100000 #@param {type:"integer"}

from pathlib import Path
if not 1 <= MES <= 12 or min(NUEVAS_PARTIDAS, MAX_FILAS, MAX_ARCHIVOS, NODOS_STOCKFISH) < 1:
    raise ValueError("Comprueba el mes y los límites positivos.")

LOCAL_DIR = Path("/content/Jade")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR = None
if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP_DIR = Path("/content/drive/MyDrive/Jade/copias")
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = LOCAL_DIR / f"jade_{RITMO}.sqlite"
print("Ritmo:", RITMO, "| Nuevas partidas por lote:", NUEVAS_PARTIDAS)
print("Copias:", BACKUP_DIR if BACKUP_DIR else "Solo temporales en Colab")


## 3 · Cargar el programa
Ejecuta esta celda. Contiene la base de datos, lectura remota, selección humana y
tablero. Está plegada para facilitar el uso; puedes abrirla para consultar o editar
todo el código Python, con comentarios en español.


In [ ]:
#@title 3. Código completo de Jade (ejecuta sin editar)
"""Jade 1.0: Stockfish MultiPV + Boltzmann + frecuencias humanas persistentes.

Código que se incluye íntegro en Jade_Colab.ipynb. No necesita este archivo aparte.
"""
import gzip
import hashlib
import html
import io
import json
import math
import os
import shutil
import sqlite3
import tempfile
import time
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import chess
import chess.engine
import chess.pgn
import chess.svg
import numpy as np
import pyarrow.parquet as pq
from huggingface_hub import HfApi, HfFileSystem
import ipywidgets as widgets
from IPython.display import display

PROFILES = (1200, 1300, 1400, 1500, 1600)
EVENTS = {"rapid": "Rated Rapid game", "blitz": "Rated Blitz game"}
REPO = "Lichess/standard-chess-games"
MAX_PLIES = 100


def position_key(board):
    # Comparte frecuencias entre transposiciones; el motor sí recibe el historial.
    return " ".join(board.fen(en_passant="legal").split()[:4])


def rating_band(rating):
    return 100 * math.floor((int(rating) + 50) / 100)


class JadeMemory:
    """SQLite evita mantener millones de posiciones en la RAM de Python."""
    def __init__(self, path, rhythm="rapid"):
        if rhythm not in EVENTS:
            raise ValueError("El ritmo debe ser rapid o blitz.")
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.rhythm = rhythm
        self.conn = sqlite3.connect(str(self.path))
        self.conn.execute("PRAGMA journal_mode=DELETE")
        self.conn.execute("PRAGMA synchronous=FULL")
        self.conn.execute("PRAGMA cache_size=-32000")
        self.conn.executescript("""
            CREATE TABLE IF NOT EXISTS meta (key TEXT PRIMARY KEY, value TEXT);
            CREATE TABLE IF NOT EXISTS counts (
                profile INTEGER, position TEXT, move TEXT, n INTEGER NOT NULL,
                PRIMARY KEY(profile, position, move)) WITHOUT ROWID;
            CREATE TABLE IF NOT EXISTS games (id TEXT PRIMARY KEY) WITHOUT ROWID;
            CREATE TABLE IF NOT EXISTS cursors (
                source TEXT PRIMARY KEY, row_group INTEGER, row_offset INTEGER,
                done INTEGER DEFAULT 0);
            CREATE TABLE IF NOT EXISTS profile_stats (
                profile INTEGER PRIMARY KEY, games INTEGER, moves INTEGER);
        """)
        config = {"schema": 1, "rhythm": rhythm, "profiles": list(PROFILES),
                  "max_plies": MAX_PLIES, "position_key": "fen4-legalep"}
        previous = self.get_meta("config")
        if previous is not None and previous != config:
            self.conn.close()
            raise ValueError("La memoria tiene otra configuración. Usa una carpeta nueva.")
        with self.conn:
            self.set_meta("config", config)

    def get_meta(self, key, default=None):
        row = self.conn.execute("SELECT value FROM meta WHERE key=?", (key,)).fetchone()
        return json.loads(row[0]) if row else default

    def set_meta(self, key, value):
        self.conn.execute("INSERT OR REPLACE INTO meta VALUES (?,?)",
                          (key, json.dumps(value, ensure_ascii=False)))

    def cursor(self, source):
        row = self.conn.execute(
            "SELECT row_group,row_offset,done FROM cursors WHERE source=?",
            (source,)).fetchone()
        return tuple(row) if row else (0, 0, 0)

    def set_cursor(self, source, group, offset, done=False):
        self.conn.execute("INSERT OR REPLACE INTO cursors VALUES (?,?,?,?)",
                          (source, int(group), int(offset), int(done)))

    def ingest(self, row):
        """Se invoca dentro de la transacción que también guarda el cursor."""
        if row.get("Event") != EVENTS[self.rhythm]:
            return "otro_ritmo"
        if row.get("WhiteTitle") == "BOT" or row.get("BlackTitle") == "BOT":
            return "bot"
        try:
            ratings = {chess.WHITE: int(row["WhiteElo"]),
                       chess.BLACK: int(row["BlackElo"])}
        except (KeyError, TypeError, ValueError, OverflowError):
            return "rating_invalido"
        if not any(rating_band(e) in PROFILES for e in ratings.values()):
            return "otro_rating"
        text = row.get("movetext")
        site = row.get("Site")
        if not isinstance(text, str) or not text.strip() or not isinstance(site, str) or not site:
            return "datos_incompletos"
        # Site es el identificador público de la partida; no guardamos jugadores.
        game_id = site.rstrip("/")
        if self.conn.execute("SELECT 1 FROM games WHERE id=?", (game_id,)).fetchone():
            return "duplicada"
        try:
            game = chess.pgn.read_game(io.StringIO(text))
        except (ValueError, IndexError):
            return "pgn_invalido"
        if game is None or game.errors or game.board().fen() != chess.STARTING_FEN:
            return "pgn_invalido"
        board = game.board()
        observations, per_profile = Counter(), Counter()
        for ply, move in enumerate(game.mainline_moves()):
            if ply >= MAX_PLIES:
                break
            band = rating_band(ratings[board.turn])
            if band in PROFILES:
                observations[(band, position_key(board), move.uci())] += 1
                per_profile[band] += 1
            board.push(move)
        if not observations:
            return "sin_movimientos"
        self.conn.execute("INSERT INTO games VALUES (?)", (game_id,))
        self.conn.executemany("""
            INSERT INTO counts VALUES (?,?,?,?)
            ON CONFLICT(profile,position,move) DO UPDATE SET n=n+excluded.n
        """, ((*key, count) for key, count in observations.items()))
        self.conn.executemany("""
            INSERT INTO profile_stats VALUES (?,1,?)
            ON CONFLICT(profile) DO UPDATE SET games=games+1, moves=moves+excluded.moves
        """, per_profile.items())
        self.set_meta("accepted_games", self.get_meta("accepted_games", 0) + 1)
        return "nueva"

    def counts_for(self, board, profile):
        return dict(self.conn.execute(
            "SELECT move,n FROM counts WHERE profile=? AND position=?",
            (rating_band(profile), position_key(board))).fetchall())

    def summary(self):
        rows = self.conn.execute("SELECT profile,games,moves FROM profile_stats ORDER BY profile").fetchall()
        return {"partidas": self.get_meta("accepted_games", 0),
                "filas_examinadas": self.get_meta("scanned_rows", 0),
                "perfiles": {int(e): {"partidas": g, "movimientos": m} for e, g, m in rows},
                "base_MB": round(self.path.stat().st_size / 1024**2, 2)}

    def close(self):
        self.conn.close()


def restore_memory(local_path, backup_dir, rhythm):
    """Restaura la copia completa más reciente; ignora archivos .part incompletos."""
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.exists() or backup_dir is None:
        return
    backups = sorted(Path(backup_dir).glob(f"jade_{rhythm}_*.sqlite.gz"), reverse=True)
    errors = []
    for backup in backups:
        temp = local_path.with_suffix(".restore")
        try:
            with gzip.open(backup, "rb") as source, temp.open("wb") as target:
                shutil.copyfileobj(source, target)
            conn = sqlite3.connect(str(temp))
            try:
                valid = conn.execute("PRAGMA quick_check").fetchone()[0] == "ok"
                config = json.loads(conn.execute("SELECT value FROM meta WHERE key='config'").fetchone()[0])
                valid = valid and config["rhythm"] == rhythm and config["schema"] == 1
            finally:
                conn.close()
            if not valid:
                raise ValueError("Copia no válida")
            os.replace(temp, local_path)
            print("Memoria recuperada:", backup.name)
            return
        except (OSError, ValueError, EOFError, sqlite3.Error, TypeError) as exc:
            errors.append(f"{backup.name}: {exc}")
            temp.unlink(missing_ok=True)
    if backups:
        raise RuntimeError("No se pudo recuperar ninguna copia. " + "; ".join(errors))


def save_memory(memory, backup_dir):
    """Snapshot consistente. Drive recibe un solo archivo comprimido por guardado."""
    if memory.conn.in_transaction:
        raise RuntimeError("Hay una transacción abierta; no se puede guardar todavía.")
    target_dir = Path(backup_dir) if backup_dir is not None else memory.path.parent / "copias"
    target_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    name = f"jade_{memory.rhythm}_{stamp}.sqlite.gz"
    final = target_dir / name
    part = target_dir / (name + ".part")
    with tempfile.TemporaryDirectory() as folder:
        snapshot = Path(folder) / "memory.sqlite"
        destination = sqlite3.connect(str(snapshot))
        try:
            memory.conn.backup(destination)
        finally:
            destination.close()
        compressed = Path(folder) / name
        with snapshot.open("rb") as source, gzip.open(compressed, "wb", compresslevel=3) as dest:
            shutil.copyfileobj(source, dest)
        shutil.copyfile(compressed, part)
        if part.stat().st_size != compressed.stat().st_size:
            raise IOError("Copia incompleta: se conserva el guardado anterior.")
        os.replace(part, final)
    # Conserva tres generaciones del mismo ritmo. Nunca borra la memoria local.
    for old in sorted(target_dir.glob(f"jade_{memory.rhythm}_*.sqlite.gz"), reverse=True)[3:]:
        old.unlink()
    place = "Drive" if backup_dir is not None else "Colab (temporal)"
    print(f"Guardado en {place}: {final.name} ({final.stat().st_size / 1024**2:.1f} MB)")
    return final


def month_files(memory, year, month):
    """Una revisión y un orden fijos permiten reanudar sin saltos ni duplicados."""
    key = f"manifest:{int(year)}-{int(month):02d}"
    manifest = memory.get_meta(key)
    if manifest is not None:
        return manifest
    api = HfApi()
    revision = memory.get_meta("dataset_revision") or api.dataset_info(REPO).sha
    folder = f"data/year={int(year)}/month={int(month):02d}"
    files = [{"path": item.path, "bytes": item.size}
             for item in api.list_repo_tree(REPO, path_in_repo=folder,
                                           repo_type="dataset", revision=revision)
             if item.path.endswith(".parquet")]
    if not files:
        raise ValueError("No hay Parquet en ese mes de la revisión fijada.")
    # Mezcla determinista de archivos. No pretende ser una muestra representativa.
    files.sort(key=lambda item: hashlib.sha256(("jade-v1:" + item["path"]).encode()).hexdigest())
    manifest = {"revision": revision, "files": files}
    with memory.conn:
        memory.set_meta("dataset_revision", revision)
        memory.set_meta(key, manifest)
    return manifest


def feed_jade(memory, year=2024, month=1, new_games=2000, max_rows=50_000,
              max_files=3, backup_dir=None, checkpoint_seconds=120,
              file_entries=None, log=print):
    """Añade partidas nuevas. Cada lote guarda contadores, IDs y cursor juntos.

    file_entries permite pruebas y Parquet locales del mismo esquema. Por defecto
    se lee Hugging Face mediante peticiones de rangos, sin descargar archivos enteros.
    """
    if min(new_games, max_rows, max_files) < 1:
        raise ValueError("Los límites deben ser positivos.")
    session = Counter()
    saved_at = time.monotonic()
    fs = None
    try:
        if file_entries is None:
            manifest = month_files(memory, year, month)
            revision = manifest["revision"]
            entries = manifest["files"]
            fs = HfFileSystem()
            log(f"Dataset: {REPO} | revisión {revision[:12]} | {year}-{month:02d}")
        else:
            entries, revision = file_entries, "local"
        opened = 0
        for entry in entries:
            name = entry["path"]
            source = f"{REPO}@{revision}/{name}"
            group, offset, done = memory.cursor(source)
            if done:
                continue
            if opened >= max_files or session["nueva"] >= new_games or session["filas"] >= max_rows:
                break
            opened += 1
            log(f"Archivo {opened}: {Path(name).name}; bloque {group}, fila {offset}")
            handle = (fs.open(f"datasets/{REPO}/{name}", "rb", revision=revision,
                              block_size=2 * 1024**2) if fs else open(name, "rb"))
            with handle:
                parquet = pq.ParquetFile(handle)
                required = ["Site", "Event", "WhiteElo", "BlackElo", "movetext"]
                missing = set(required) - set(parquet.schema_arrow.names)
                if missing:
                    raise ValueError(f"Esquema incompatible: faltan {sorted(missing)}")
                columns = required + [x for x in ("WhiteTitle", "BlackTitle")
                                      if x in parquet.schema_arrow.names]
                for rg in range(group, parquet.num_row_groups):
                    start = offset if rg == group else 0
                    total = parquet.metadata.row_group(rg).num_rows
                    batch_start = 0
                    for batch in parquet.iter_batches(batch_size=128, row_groups=[rg], columns=columns):
                        batch_end = batch_start + batch.num_rows
                        if batch_end <= start:
                            batch_start = batch_end
                            continue
                        rows = batch.to_pylist()
                        local_start = max(0, start - batch_start)
                        delta = Counter()
                        next_offset = batch_start + local_start
                        # Una interrupción revierte tanto frecuencias como cursor de este lote.
                        with memory.conn:
                            for index in range(local_start, len(rows)):
                                if (session["nueva"] + delta["nueva"] >= new_games or
                                    session["filas"] + delta["filas"] >= max_rows):
                                    break
                                result = memory.ingest(rows[index])
                                delta[result] += 1
                                delta["filas"] += 1
                                next_offset = batch_start + index + 1
                            completed = next_offset >= total
                            next_group = rg + 1 if completed else rg
                            memory.set_cursor(source, next_group, 0 if completed else next_offset,
                                              done=next_group >= parquet.num_row_groups)
                            memory.set_meta("scanned_rows", memory.get_meta("scanned_rows", 0) + delta["filas"])
                        session.update(delta)
                        batch_start = batch_end
                        if session["filas"] // 2048 != (session["filas"] - delta["filas"]) // 2048:
                            log(f"Leídas {session['filas']:,} | nuevas {session['nueva']:,}")
                        if time.monotonic() - saved_at >= checkpoint_seconds:
                            save_memory(memory, backup_dir)
                            saved_at = time.monotonic()
                        if session["nueva"] >= new_games or session["filas"] >= max_rows:
                            return dict(session)
        if opened == 0:
            log("Mes agotado. Cambia el mes para añadir otra fuente.")
        return dict(session)
    except KeyboardInterrupt:
        log("Carga interrumpida. Se conserva lo confirmado hasta el último lote.")
        return dict(session)
    finally:
        # Incluso ante un error de red, se guarda el último lote confirmado.
        save_memory(memory, backup_dir)


"""Jade 1.2: candidatas humanas y de motor, presupuesto de tiempo y muestreo."""
import math
import time
import numpy as np
import chess
import chess.engine


class JadeEngine:
    VERSION = "1.2"

    def __init__(self, binary, memory, nodes=100_000):
        self.binary, self.memory, self.nodes = str(binary), memory, int(nodes)
        self.engine = chess.engine.SimpleEngine.popen_uci(self.binary, timeout=60)
        self.engine.configure({"Threads": 1, "Hash": 128, "Skill Level": 20,
                               "UCI_LimitStrength": False})
        self.rng = np.random.default_rng()
        self.game_token = object()

    def new_game(self):
        self.game_token = object()
        self.engine.configure({"Clear Hash": None})

    @staticmethod
    def candidate_limit(seconds=None, maximum=15):
        if maximum not in (12, 15):
            raise ValueError("El máximo de candidatas debe ser 12 o 15.")
        if seconds is None or seconds > 180:
            return maximum
        if seconds > 90:
            return min(maximum, 12)
        if seconds > 30:
            return 8
        if seconds > 10:
            return 5
        return 3

    @staticmethod
    def search_seconds(remaining=None):
        # Presupuesto TOTAL de búsqueda, incluidas las comprobaciones humanas.
        if remaining is None or remaining > 180:
            return .9
        if remaining > 90:
            return .6
        if remaining > 30:
            return .35
        if remaining > 10:
            return .18
        return max(.005, min(.08, remaining * .10))

    def human_counts(self, board, profile):
        legal = {m.uci() for m in board.legal_moves}
        return {uci: int(n) for uci, n in self.memory.counts_for(board, profile).items()
                if uci in legal and n > 0}

    @staticmethod
    def _row(board, info, counts):
        move = info["pv"][0]
        score = info["score"].pov(board.turn)
        return dict(move=move, uci=move.uci(), san=board.san(move),
                    cp=score.score(), mate=score.mate(), value=score.score(mate_score=100000),
                    human_count=counts.get(move.uci(), 0), depth=info.get("depth", 0))

    def candidates(self, board, profile=1300, remaining=None, maximum=15, use_humans=True):
        if not board.is_valid():
            raise ValueError("Posición no válida.")
        if board.is_game_over():
            return []
        k = min(self.candidate_limit(remaining, maximum), board.legal_moves.count())
        counts = self.human_counts(board, profile) if use_humans else {}
        # Se reservan plazas a jugadas reales, incluso fuera del ranking del motor.
        ranked = sorted(counts, key=lambda uci: (-counts[uci], uci))
        opening = board.ply() < 24
        known = [chess.Move.from_uci(uci) for uci in ranked if counts[uci] >= 2]
        human_slots = min(k - 1, (k * 2 // 3) if opening else (k // 2))
        humans = known[:human_slots]
        budget = self.search_seconds(remaining)
        begun = time.monotonic()
        if not humans:
            infos = self.engine.analyse(board, chess.engine.Limit(time=budget, nodes=self.nodes),
                                        multipv=k, game=self.game_token)
            rows = [self._row(board, info, counts) for info in infos if info.get("pv") and "score" in info]
            for row in rows:
                row["outside_engine_list"] = False
            return sorted(rows, key=lambda row: row["value"], reverse=True)

        # Primera búsqueda libre: conserva una referencia táctica y completa plazas.
        preliminary = self.engine.analyse(board, chess.engine.Limit(time=budget * .35, nodes=max(100, self.nodes // 3)),
                                          multipv=k, game=self.game_token)
        engine_moves = [info["pv"][0] for info in preliminary if info.get("pv")]
        if not engine_moves:
            raise RuntimeError("Stockfish no devolvió candidatas.")
        pool = list(dict.fromkeys([engine_moves[0]] + humans + engine_moves))[:k]
        # Un segundo análisis compara TODAS las candidatas, incluidas las humanas.
        left = max(.005, budget - (time.monotonic() - begun))
        infos = self.engine.analyse(board, chess.engine.Limit(time=left, nodes=max(100, self.nodes * 2 // 3)),
                                    multipv=len(pool), root_moves=pool, game=self.game_token)
        rows = [self._row(board, info, counts) for info in infos if info.get("pv") and "score" in info]
        for row in rows:
            row["outside_engine_list"] = row["move"] not in engine_moves
        return sorted(rows, key=lambda row: row["value"], reverse=True)

    def distribution(self, board, candidates, profile=1300, beta=1.0, temperature=None, remaining=None):
        if profile not in (1200, 1300, 1400, 1500, 1600):
            raise ValueError("Perfil no disponible.")
        t = float(np.interp(profile, [1200, 1600], [180., 45.])) if temperature is None else float(temperature)
        if not math.isfinite(t) or t <= 0 or not math.isfinite(beta) or not 0 <= beta <= 2:
            raise ValueError("Temperatura positiva y peso humano entre 0 y 2.")
        if not candidates:
            raise ValueError("No hay candidatas.")
        # Con poco tiempo se reduce el cálculo y aumenta la dispersión, aunque
        # haya menos candidatas. No se obliga a escoger siempre la primera.
        pressure = 1. if remaining is None or remaining > 90 else 1.2 if remaining > 30 else 1.5 if remaining > 10 else 1.9
        t *= pressure
        counts = self.human_counts(board, profile)
        values = np.array([row["value"] for row in candidates], dtype=float)
        human = np.array([counts.get(row["uci"], 0) for row in candidates], dtype=float)
        loss = np.maximum(0., values.max() - values)
        weights = np.exp(np.clip((values - values.max()) / t, -700, 0))
        p_engine = weights / weights.sum()
        opening = board.ply() < 24
        # Tolerancia heurística: admite gambitos y elecciones subóptimas comunes.
        free_cp = float(np.interp(profile, [1200, 1600], [220., 100.])) * (1.25 if opening else 1.)
        penalty = np.exp(-np.maximum(0., loss - free_cp) / (120. * pressure))
        best_mate = candidates[int(np.argmax(values))]["mate"]
        for i, row in enumerate(candidates):
            if row["mate"] is not None and row["mate"] < 0 and (best_mate is None or best_mate >= 0):
                penalty[i] = 0.  # Evita un mate detectado si hay una alternativa.
            elif best_mate is not None and best_mate > 0 and row["mate"] is None:
                # No convierte un mate positivo en una diferencia ficticia de cp.
                penalty[i] = math.exp(-max(0., 350. - (row["cp"] or 0)) / 200.)
        evidence = float(human.sum())
        p_human = human * penalty
        mix = ((beta / (beta + .25)) * (evidence / (evidence + 15.)) * (1. if opening else .7)) if beta else 0.
        if p_human.sum() <= 0:
            mix, p_human = 0., p_engine.copy()
        else:
            p_human /= p_human.sum()
        probabilities = (1. - mix) * p_engine + mix * p_human
        details = dict(temperature=t, human_mix=float(mix), candidate_count=len(candidates),
                       candidate_observations=int(evidence), position_observations=sum(counts.values()),
                       # Alias para abrir partidas guardadas por la interfaz anterior.
                       top5_observations=int(evidence), opening=opening)
        return probabilities, details

    def reaction_seconds(self, board, candidates, details, remaining=None, pace=1.):
        if pace < 0 or not math.isfinite(pace):
            raise ValueError("El ritmo de respuesta debe ser positivo.")
        if board.legal_moves.count() == 1:
            target = self.rng.uniform(.4, .9)
        elif remaining is not None and remaining <= 10:
            target = self.rng.uniform(.15, .55)
        elif remaining is not None and remaining <= 30:
            target = self.rng.uniform(.4, 1.3)
        elif remaining is not None and remaining <= 90:
            target = self.rng.uniform(1., 3.)
        else:
            target = self.rng.uniform(2.5, 6.)
            close = sum(candidates[0]["value"] - row["value"] < 70 for row in candidates)
            target *= 1. + min(close, 5) * .10
            if details["opening"] and details["position_observations"] >= 30:
                target *= .75
        target = min(10., target * pace)
        if remaining is not None:
            target = min(target, max(0., remaining * .18))
        return float(target)

    def choose(self, board, profile=1300, beta=1., temperature=None, remaining=None, maximum=15, pace=1.):
        candidates = self.candidates(board, profile, remaining, maximum, use_humans=beta > 0)
        p, details = self.distribution(board, candidates, profile, beta, temperature, remaining)
        index = int(self.rng.choice(len(candidates), p=p))
        row = candidates[index]
        details.update(selected_san=row["san"], selected_human_count=row["human_count"],
                       outside_engine_list=row["outside_engine_list"], probability=float(p[index]),
                       max_candidates=self.candidate_limit(remaining, maximum))
        details["reaction_seconds"] = self.reaction_seconds(board, candidates, details, remaining, pace)
        return row["move"], details

    def close(self):
        try:
            self.engine.quit()
        except chess.engine.EngineTerminatedError:
            pass

"""Interfaz Jade 1.2: tablero SVG, reloj de práctica y controles táctiles.

No cambia la base de frecuencias ni entrena con las jugadas del usuario.
El constructor también funciona sin Colab para verificar la lógica y el HTML.
"""
import html
import json
import math
import os
import tempfile
import time
import uuid
from pathlib import Path

import chess
import chess.pgn
import chess.svg
from IPython.display import HTML, JSON, display


class JadeBoard:
    VERSION = "1.2"
    PROFILES = (1200, 1300, 1400, 1500, 1600)

    def __init__(self, jade, game_path=None, previous=None, now=None, sleeper=None):
        self.jade = jade
        self.board = chess.Board()
        self.human, self.orientation = chess.WHITE, chess.WHITE
        self.profile, self.beta = 1300, 1.0
        self.selected, self.pending = None, []
        self.finished, self.busy, self.closed, self.started = False, False, False, False
        self.result, self.message = "*", "Elige tu color y pulsa Jugar."
        self.last_details = None
        self.now, self.sleep = now or time.monotonic, sleeper or time.sleep
        self.initial_seconds, self.increment = 600., 0.
        self.remaining = [600., 600.]  # Índices: negras=0, blancas=1.
        self.paused, self.clock_started = False, None
        self.clock_history, self.move_clocks = [], []
        self.maximum, self.pace = 15, 1.
        self.revision = 0
        self.callback_name = "jade.ui." + uuid.uuid4().hex
        self.game_path = Path(game_path) if game_path is not None else None
        self.save_note = "La partida solo se conserva en esta sesión."
        self.load_error = None
        if previous is not None:
            if hasattr(previous, "settle_clock"):
                previous.settle_clock()
            # Conserva el historial, también al actualizar desde el tablero 1.0.
            self.board = previous.board.copy(stack=True)
            self.human = bool(previous.human)
            self.orientation = getattr(previous, "orientation", self.human)
            self.profile, self.beta = previous.profile, previous.beta
            self.finished, self.result = previous.finished, previous.result
            self.last_details = previous.last_details
            self.started = getattr(previous, "started", True)
            if hasattr(previous, "initial_seconds"):
                self.initial_seconds, self.increment = previous.initial_seconds, previous.increment
                self.remaining = list(previous.remaining)
                self.clock_history = [list(x) for x in previous.clock_history]
                self.move_clocks = list(previous.move_clocks)
                self.maximum, self.pace = previous.maximum, previous.pace
                self.paused = bool(self.started and not self.finished)
            else:
                # A una partida de 1.0/1.1 no se le inventa tiempo ya consumido.
                self.initial_seconds, self.remaining = 0., [0., 0.]
                self.clock_history = [[0., 0.] for _ in self.board.move_stack]
                self.move_clocks = [None for _ in self.board.move_stack]
            self.load_error = getattr(previous, "load_error", None)
            if self.load_error:
                self.message = self.load_error
                self.save_note = "El archivo original se conserva."
            else:
                self.message = "Partida conservada. Pulsa Continuar reloj." if self.paused else "Interfaz actualizada. Tu partida se conserva."
                self.save_game()
        elif self.game_path is not None and self.game_path.exists():
            try:
                self.restore_game(json.loads(self.game_path.read_text(encoding="utf-8")))
                self.message = "Partida recuperada en pausa. Pulsa Continuar reloj." if self.paused else "Partida recuperada. Puedes continuar."
                self.save_note = "Partida recuperada de la copia guardada."
            except (OSError, ValueError, KeyError, TypeError) as exc:
                self.load_error = "No se pudo recuperar la partida: " + str(exc)
                self.message = self.load_error
                self.save_note = "El archivo original se conserva."

    def restore_game(self, data):
        if data["schema"] not in (1, 2) or data["rhythm"] != self.jade.memory.rhythm:
            raise ValueError("La copia tiene otro formato o ritmo.")
        board = chess.Board(data["root_fen"])
        if not board.is_valid():
            raise ValueError("Posición inicial no válida.")
        for uci in data["moves"]:
            board.push_uci(uci)
        profile, beta = int(data["profile"]), float(data["beta"])
        if profile not in self.PROFILES or not math.isfinite(beta) or not 0 <= beta <= 2:
            raise ValueError("Ajustes de partida no válidos.")
        if data["result"] not in ("*", "1-0", "0-1", "1/2-1/2"):
            raise ValueError("Resultado no válido.")
        self.board, self.profile, self.beta = board, profile, beta
        self.human = bool(data["human"])
        self.orientation = bool(data.get("orientation", self.human))
        self.started, self.finished = bool(data["started"]), bool(data["finished"])
        self.result = data["result"]
        self.last_details = data.get("last_details")
        clock = data.get("clock", {})
        self.initial_seconds = float(clock.get("initial", 0))
        self.increment = float(clock.get("increment", 0))
        self.remaining = [float(x) for x in clock.get("remaining", [0., 0.])]
        if len(self.remaining) != 2 or any(not math.isfinite(x) or x < 0 for x in self.remaining):
            raise ValueError("Reloj guardado no válido.")
        self.clock_history = clock.get("history", [[0., 0.] for _ in board.move_stack])
        self.move_clocks = clock.get("moves", [None for _ in board.move_stack])
        if len(self.clock_history) != len(board.move_stack) or len(self.move_clocks) != len(board.move_stack):
            raise ValueError("Historial del reloj incompleto.")
        self.maximum, self.pace = data.get("maximum", 15), float(data.get("pace", 1.))
        if self.maximum not in (12, 15) or self.pace not in (.5, 1., 1.5):
            raise ValueError("Configuración del selector no válida.")
        self.paused = bool(self.initial_seconds and self.started and not self.finished)
        self.clock_started = None
        self.jade.new_game()
        self.update_outcome()

    def game_data(self):
        return dict(schema=2, rhythm=self.jade.memory.rhythm,
                    root_fen=self.board.root().fen(),
                    moves=[move.uci() for move in self.board.move_stack],
                    human=self.human, orientation=self.orientation, profile=self.profile,
                    beta=self.beta, started=self.started, finished=self.finished,
                    result=self.result, last_details=self.last_details,
                    maximum=self.maximum, pace=self.pace,
                    clock=dict(initial=self.initial_seconds, increment=self.increment,
                               remaining=self.remaining, history=self.clock_history, moves=self.move_clocks))

    def clock_running(self):
        return bool(self.initial_seconds and self.started and not self.finished and not self.paused and not self.closed)

    def settle_clock(self):
        if not self.clock_running() or self.clock_started is None:
            return
        stamp = self.now()
        side = int(self.board.turn)
        self.remaining[side] = max(0., self.remaining[side] - max(0., stamp - self.clock_started))
        self.clock_started = stamp
        if self.remaining[side] <= 0:
            opponent = not self.board.turn
            self.finished = True
            self.result = "1/2-1/2" if self.board.has_insufficient_material(opponent) else ("1-0" if opponent else "0-1")
            self.message = ("Tu tiempo se ha agotado" if self.board.turn == self.human else "Jade se ha quedado sin tiempo") + " · " + self.result
            self.clock_started = None
            self.pending = []
            self.save_game()

    def clock_view(self):
        values = list(self.remaining)
        if self.clock_running() and self.clock_started is not None:
            side = int(self.board.turn)
            values[side] = max(0., values[side] - max(0., self.now() - self.clock_started))
        return values

    def commit_move(self, move):
        self.settle_clock()
        if self.finished:
            return False
        self.clock_history.append(list(self.remaining))
        mover = int(self.board.turn)
        if self.initial_seconds:
            self.remaining[mover] += self.increment
        self.move_clocks.append(self.remaining[mover] if self.initial_seconds else None)
        self.board.push(move)
        self.clock_started = self.now() if self.clock_running() else None
        return True

    def save_game(self):
        if self.game_path is None:
            return
        temporary = None
        try:
            self.game_path.parent.mkdir(parents=True, exist_ok=True)
            with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", suffix=".part",
                                             dir=self.game_path.parent, delete=False) as out:
                temporary = Path(out.name)
                json.dump(self.game_data(), out, ensure_ascii=False)
                out.flush()
                os.fsync(out.fileno())
            os.replace(temporary, self.game_path)
            place = "Drive" if str(self.game_path).startswith("/content/drive/") else "el almacenamiento configurado"
            self.save_note = "Partida guardada en " + place + "."
        except OSError as exc:
            self.save_note = "No se pudo guardar la partida. Descarga el PGN antes de salir. " + str(exc)
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)

    def archive_game(self):
        """Al empezar otra, conserva la anterior como PGN si tiene movimientos."""
        if self.game_path is not None and self.board.move_stack:
            path = self.game_path.parent / ("partida_" + uuid.uuid4().hex + ".pgn")
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(self.pgn(), encoding="utf-8")

    def update_outcome(self):
        outcome = self.board.outcome()
        if outcome is not None:
            self.finished, self.result = True, outcome.result()
            reason = {"CHECKMATE": "Jaque mate", "STALEMATE": "Tablas por ahogado",
                      "INSUFFICIENT_MATERIAL": "Tablas por material insuficiente",
                      "SEVENTYFIVE_MOVES": "Tablas por 75 movimientos",
                      "FIVEFOLD_REPETITION": "Tablas por repetición"}
            self.message = reason.get(outcome.termination.name, "Partida terminada") + " · " + self.result

    def pgn(self):
        game = chess.pgn.Game.from_board(self.board)
        game.headers.update({"Event": "Práctica con Jade", "White": "Tú" if self.human else "Jade",
                             "Black": "Jade" if self.human else "Tú", "Result": self.result,
                             "JadeProfile": str(self.profile),
                             "TimeControl": f"{int(self.initial_seconds)}+{int(self.increment)}" if self.initial_seconds else "-"})
        for node, clock in zip(game.mainline(), self.move_clocks):
            if clock is not None:
                node.set_clock(clock)
        return str(game)

    def svg(self):
        targets = {move.to_square for move in self.board.legal_moves if move.from_square == self.selected}
        svg = chess.svg.board(
            self.board, orientation=self.orientation, coordinates=False,
            lastmove=self.board.peek() if self.board.move_stack else None,
            check=self.board.king(self.board.turn) if self.board.is_check() else None,
            fill={self.selected: "#f5cc69"} if self.selected is not None else {},
            colors={"square light": "#edf1df", "square dark": "#739889",
                    "square light lastmove": "#d9dc8a", "square dark lastmove": "#b7c46c"})
        # El SVG sin coordenadas usa casillas de 45 unidades. Las piezas proceden
        # de python-chess; estas marcas solo indican los destinos legales.
        dots = []
        for square in targets:
            file, rank = chess.square_file(square), chess.square_rank(square)
            col = file if self.orientation else 7 - file
            row = 7 - rank if self.orientation else rank
            x, y = 45 * col + 22.5, 45 * row + 22.5
            if self.board.piece_at(square):
                dots.append(f'<circle cx="{x}" cy="{y}" r="19" fill="none" stroke="#153f32" stroke-width="3" opacity=".7"/>')
            else:
                dots.append(f'<circle cx="{x}" cy="{y}" r="6" fill="#153f32" opacity=".45"/>')
        return svg.replace("</svg>", "".join(dots) + "</svg>")

    def state(self):
        self.update_outcome()
        files = list("abcdefgh") if self.orientation else list("hgfedcba")
        ranks = list(range(8, 0, -1)) if self.orientation else list(range(1, 9))
        names = {chess.PAWN: "peón", chess.KNIGHT: "caballo", chess.BISHOP: "alfil",
                 chess.ROOK: "torre", chess.QUEEN: "dama", chess.KING: "rey"}
        targets = {m.to_square for m in self.board.legal_moves if m.from_square == self.selected}
        squares = []
        for rank in ranks:
            for file in files:
                name = file + str(rank)
                sq = chess.parse_square(name)
                piece = self.board.piece_at(sq)
                label = name + (": " + names[piece.piece_type] + (" blanco" if piece.color else " negro") if piece else ": vacía")
                squares.append(dict(name=name, label=label, selected=sq == self.selected, target=sq in targets))
        replay, history = self.board.root(), []
        for move in self.board.move_stack:
            number, white = replay.fullmove_number, replay.turn
            san = replay.san(move)
            if white or not history:
                history.append([number, san if white else "…", ""])
            if not white:
                history[-1][2] = san
            replay.push(move)
        ai_pending = self.started and not self.finished and not self.paused and self.board.turn != self.human and not self.closed
        return dict(revision=self.revision, svg=self.svg(), squares=squares, files=files, ranks=ranks,
                    message=self.message, started=self.started, finished=self.finished, closed=self.closed,
                    human=self.human, orientation=self.orientation, profile=self.profile, beta=self.beta, rhythm=self.jade.memory.rhythm,
                    result=self.result, history=history, pgn=self.pgn(), plies=len(self.board.move_stack),
                    can_move=self.started and not self.finished and not self.closed and not self.paused and self.board.turn == self.human,
                    can_undo=bool(self.board.move_stack) and not (not self.human and len(self.board.move_stack) == 1),
                    can_draw=self.started and not self.finished and self.board.turn == self.human and self.board.can_claim_draw(),
                    ai_pending=ai_pending, promotion=bool(self.pending), save_note=self.save_note,
                    observations=(self.last_details or {}).get("candidate_observations", (self.last_details or {}).get("top5_observations")),
                    decision=self.last_details, maximum=self.maximum, pace=self.pace,
                    clock=dict(enabled=bool(self.initial_seconds), remaining=self.clock_view(),
                               active=int(self.board.turn) if self.clock_running() else None,
                               paused=self.paused, initial=self.initial_seconds, increment=self.increment),
                    turn=("Partida terminada" if self.finished else "En pausa" if self.paused else "Tu turno" if self.board.turn == self.human else "Turno de Jade"))

    def push_human(self, move):
        if move not in self.board.legal_moves:
            self.message = "Esa jugada no es legal. Selecciona un destino marcado."
            return
        self.message = "Has jugado " + self.board.san(move) + "."
        if not self.commit_move(move):
            return
        self.selected, self.pending = None, []
        self.update_outcome()
        self.save_game()  # Se guarda también antes de que Stockfish responda.

    def ai_turn(self):
        if not self.started or self.finished or self.paused or self.board.turn == self.human:
            return
        if self.board.can_claim_draw():
            self.finished, self.result, self.message = True, "1/2-1/2", "Jade reclama tablas."
        else:
            try:
                began = self.now()
                seconds = self.clock_view()[int(self.board.turn)] if self.initial_seconds else None
                move, details = self.jade.choose(self.board, self.profile, self.beta,
                                                 remaining=seconds, maximum=self.maximum, pace=self.pace)
                if move not in self.board.legal_moves:
                    raise ValueError("El motor devolvió una jugada no válida.")
                delay = max(0., details.get("reaction_seconds", 0.) - (self.now() - began))
                if self.initial_seconds:
                    delay = min(delay, max(0., self.clock_view()[int(self.board.turn)] * .5))
                if delay:
                    self.sleep(delay)
                self.settle_clock()
                if not self.finished:
                    self.message = "Jade juega " + self.board.san(move) + ". Te toca."
                    if self.commit_move(move):
                        details["elapsed_seconds"] = self.now() - began
                        self.last_details = details
                        self.update_outcome()
            except Exception as exc:
                self.message = "No se pudo calcular la respuesta. Pulsa Reintentar. Detalle: " + str(exc)
        self.save_game()

    def handle(self, action, payload=None, revision=None):
        """Un único punto de entrada. Python valida todas las jugadas y el turno."""
        if self.closed:
            return dict(self.state(), message="Este tablero se ha cerrado. Usa la salida más reciente de la celda 6.")
        if self.busy:
            return self.state()
        if revision is not None and int(revision) != self.revision:
            return dict(self.state(), message="Tablero sincronizado. Repite tu acción.")
        payload = payload or {}
        self.busy = True
        try:
            self.settle_clock()
            if action == "new":
                profile, beta = int(payload.get("profile", 1300)), float(payload.get("beta", 1))
                if profile not in self.PROFILES or not math.isfinite(beta) or not 0 <= beta <= 2:
                    raise ValueError("Elige uno de los perfiles disponibles y un peso entre 0 y 2.")
                initial, increment = map(int, str(payload.get("time_control", "600+0")).split("+"))
                maximum, pace = int(payload.get("maximum", 15)), float(payload.get("pace", 1.))
                if (initial, increment) not in ((0, 0), (180, 2), (300, 0), (600, 0), (600, 5), (900, 10)) or maximum not in (12, 15) or pace not in (.5, 1., 1.5):
                    raise ValueError("Selecciona un reloj y un ritmo de respuesta válidos.")
                # Un archivo ilegible se conserva antes de crear la nueva partida.
                if self.load_error and self.game_path and self.game_path.exists():
                    recovered = self.game_path.with_name("copia_no_leida_" + uuid.uuid4().hex + ".json")
                    recovered.write_bytes(self.game_path.read_bytes())
                self.archive_game()
                self.jade.new_game()
                self.board = chess.Board()
                self.human = payload.get("color", "white") == "white"
                self.orientation = self.human
                self.profile, self.beta = profile, beta
                self.initial_seconds, self.increment = float(initial), float(increment)
                self.remaining = [float(initial), float(initial)]
                self.clock_history, self.move_clocks = [], []
                self.maximum, self.pace, self.paused = maximum, pace, False
                self.started, self.finished, self.result = True, False, "*"
                self.selected, self.pending, self.last_details = None, [], None
                self.message, self.load_error = "Toca una pieza y después un destino marcado.", None
                self.clock_started = self.now() if initial else None
                self.save_game()
            elif action == "pause" and self.started and not self.finished:
                self.paused = not self.paused
                self.clock_started = self.now() if self.clock_running() else None
                self.message = "Reloj en pausa." if self.paused else "Partida reanudada."
                self.save_game()
            elif action == "tick":
                pass  # settle_clock ya comprueba la caída de bandera.
            elif action == "flip":
                self.orientation = not self.orientation
                if self.started:
                    self.save_game()
            elif action == "undo" and self.board.move_stack:
                if not self.human and len(self.board.move_stack) == 1:
                    self.message = "Todavía no has movido."
                else:
                    while self.board.move_stack:
                        self.board.pop()
                        if self.clock_history:
                            self.remaining = self.clock_history.pop()
                        if self.move_clocks:
                            self.move_clocks.pop()
                        if self.board.turn == self.human:
                            break
                    self.finished, self.result, self.selected = False, "*", None
                    self.pending, self.last_details = [], None
                    self.message = "Turno deshecho. Puedes elegir otra jugada."
                    self.clock_started = self.now() if self.clock_running() else None
                    self.save_game()
            elif action == "resign" and self.started and not self.finished:
                self.finished, self.result = True, "0-1" if self.human else "1-0"
                self.message, self.pending = "Te has rendido. Gana Jade.", []
                self.save_game()
            elif action == "draw" and self.started and not self.finished:
                if self.board.turn == self.human and self.board.can_claim_draw():
                    self.finished, self.result, self.message = True, "1/2-1/2", "Tablas reclamadas."
                    self.pending = []
                    self.save_game()
                else:
                    self.message = "Esta posición todavía no permite reclamar tablas."
            elif action == "ai":
                self.ai_turn()
            elif action == "cancel_promotion":
                self.pending = []
                self.message = "Promoción cancelada. Puedes elegir otro movimiento."
            elif self.started and not self.finished and not self.paused and self.board.turn == self.human:
                if action == "square":
                    sq = chess.parse_square(payload["square"])
                    piece = self.board.piece_at(sq)
                    if piece and piece.color == self.human:
                        self.selected = None if self.selected == sq else sq
                        self.pending = []
                        self.message = "Elige un destino marcado." if self.selected is not None else "Toca una pieza."
                    elif self.selected is not None:
                        moves = [m for m in self.board.legal_moves if m.from_square == self.selected and m.to_square == sq]
                        if any(m.promotion for m in moves):
                            self.pending = moves
                            self.message = "Elige la pieza para promocionar."
                        elif moves:
                            self.push_human(moves[0])
                        else:
                            self.message = "Toca uno de los destinos marcados."
                elif action == "promote":
                    symbol = payload.get("piece", "q")
                    move = next((m for m in self.pending if chess.piece_symbol(m.promotion) == symbol), None)
                    if move is not None:
                        self.push_human(move)
                elif action == "uci":
                    self.push_human(chess.Move.from_uci(str(payload.get("uci", "")).strip().lower()))
        except (ValueError, KeyError, TypeError, OSError) as exc:
            self.message = "No se pudo realizar la acción: " + str(exc)
        finally:
            self.busy = False
            self.revision += 1
        return self.state()

    def callback(self, action, payload=None, revision=None):
        return JSON(self.handle(action, payload, revision))

    def html(self):
        # Evita cerrar el bloque script al incorporar textos del estado.
        initial = json.dumps(self.state(), ensure_ascii=False).replace("<", "\\u003c")
        return (JADE_HTML.replace("__JADE_CALLBACK__", self.callback_name)
                .replace("__JADE_INITIAL__", initial))

    def show(self):
        from google.colab import output
        output.register_callback(self.callback_name, self.callback)
        display(HTML(self.html()))

    def close(self):
        self.settle_clock()
        if self.started:
            self.paused, self.clock_started = True, None
            self.save_game()
        self.closed = True


JADE_HTML = r'''
<meta name="viewport" content="width=device-width, initial-scale=1">
<meta name="color-scheme" content="light">
<style>
html,body{margin:0;padding:0;width:100%;min-width:0;background:transparent}
.jade,.jade *{box-sizing:border-box}
.jade{color-scheme:light;width:100%;max-width:460px;min-width:0;margin:0 auto;padding:12px;
  background:#f7f8f3;color:#183a30;border:1px solid #dce4d8;border-radius:18px;
  font:15px/1.45 system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;overflow-wrap:anywhere}
.jade button,.jade select,.jade input,.jade textarea{font:inherit;max-width:100%;min-width:0;color:#183a30;color-scheme:light}
.jade button{border:1px solid #c7d4c8;border-radius:10px;background:#fff;padding:10px 12px;min-height:44px;
  cursor:pointer;touch-action:manipulation;-webkit-tap-highlight-color:transparent}
.jade button:disabled{cursor:default;opacity:.5}
.jade button:focus-visible,.jade summary:focus-visible,.jade select:focus-visible{outline:3px solid #216e55;outline-offset:2px}
.jade .brand{display:flex;justify-content:space-between;gap:8px;align-items:center;margin-bottom:12px}
.jade h1{font-size:29px;line-height:1.1;letter-spacing:-1px;margin:0}
.jade .muted{color:#52685b;font-size:12px}
.jade .tag{background:#e4ecdf;padding:5px 9px;border-radius:20px;font-size:12px;white-space:nowrap}
.jade .setup-grid{display:grid;grid-template-columns:1fr 1fr;gap:10px;margin:10px 0}
.jade label{display:block;font-size:13px;font-weight:600}
.jade select,.jade input[type=text],.jade textarea{width:100%;background:#fff;border:1px solid #bbcbbb;border-radius:9px;padding:10px;margin-top:4px;min-height:44px}
.jade .primary{width:100%;background:#1b624b;border-color:#1b624b;color:white;font-weight:650}
.jade details{border-top:1px solid #dce4d8;padding:10px 0}
.jade summary{cursor:pointer;font-weight:600;min-height:32px;display:list-item;margin-left:18px}
.jade .advanced{padding:8px 0}
.jade input[type=range]{width:100%;accent-color:#1b624b}
.jade .status{background:#e9eee3;border-radius:10px;padding:9px 11px;margin:12px 0 8px}
.jade #turn{font-weight:750;display:block}
.jade #message{font-size:13px;display:block;min-height:19px}
.jade .player{display:flex;align-items:center;justify-content:space-between;gap:8px;padding:7px 0;font-size:13px}
.jade .player b{font-weight:650}
.jade .clock{font:700 21px/1.1 ui-monospace,monospace;letter-spacing:1px;padding:7px 10px;border-radius:8px;background:#e4eadf;color:#3d5144;white-space:nowrap}
.jade .clock.active{background:#174d3b;color:#fff}
.jade .clock.low{background:#9d392e;color:#fff}
.jade .clock.paused{opacity:.65}
.jade .board-frame{display:grid;grid-template-columns:12px minmax(0,1fr);grid-template-rows:auto 16px;gap:2px;width:100%;min-width:0}
.jade .ranks{display:grid;grid-template-rows:repeat(8,1fr);font-size:10px;text-align:center;align-items:center;color:#52685b}
.jade .files{grid-column:2;display:grid;grid-template-columns:repeat(8,1fr);font-size:10px;text-align:center;color:#52685b}
.jade .board{position:relative;aspect-ratio:1;width:100%;min-width:0;border-radius:4px;overflow:hidden;background:#739889}
.jade #drawing{position:absolute;inset:0}
.jade #drawing svg{display:block;width:100%;height:100%;max-width:100%}
.jade .squares{position:absolute;inset:0;display:grid;grid-template-columns:repeat(8,minmax(0,1fr));grid-template-rows:repeat(8,minmax(0,1fr))}
.jade .squares button{display:block;width:100%;height:100%;min-height:0;padding:0;border:0;margin:0;border-radius:0;background:transparent!important;opacity:1!important}
.jade .squares button:focus-visible{outline:3px solid #17664b;outline-offset:-3px}
.jade .actions{display:grid;grid-template-columns:1fr 1fr;gap:8px;margin:10px 0}
.jade .actions button{font-size:13px;padding:8px}
.jade .danger{color:#9a3830;border-color:#e5b9b0}
.jade [hidden]{display:none!important}
.jade .promotion,.jade .confirm{background:#fff9e6;border:1px solid #e1c475;border-radius:10px;padding:10px;margin:10px 0}
.jade .promotions{display:grid;grid-template-columns:1fr 1fr;gap:6px;margin:8px 0}
.jade .history{max-height:150px;overflow:auto;background:#fff;border-radius:8px;margin:6px 0}
.jade table{border-collapse:collapse;width:100%;font-size:14px;text-align:left;table-layout:fixed}
.jade th,.jade td{padding:6px 10px;border-bottom:1px solid #e8eee4}
.jade th:first-child,.jade td:first-child{width:40px;color:#617264}
.jade textarea{height:110px;font:12px/1.5 ui-monospace,monospace;resize:vertical}
.jade .footer{margin-top:8px;font-size:11px;color:#607264}
@media(max-width:280px){.jade{padding:8px;border-radius:12px;font-size:14px}.jade .setup-grid{grid-template-columns:1fr}.jade .tag{font-size:10px}.jade button{padding:8px}.jade .actions{gap:5px}}
</style>
<main class="jade" id="jade">
  <header class="brand"><div><h1>Jade</h1><div class="muted">Tu espacio para practicar</div></div><span class="tag">Jade 1.2</span></header>
  <details id="setup" open><summary>Preparar partida</summary>
    <div class="setup-grid">
      <label>Perfil de Jade<select id="profile"><option>1200</option><option selected>1300</option><option>1400</option><option>1500</option><option>1600</option></select></label>
      <label>Tú juegas con<select id="color"><option value="white">Blancas</option><option value="black">Negras</option></select></label>
      <label>Reloj<select id="time-control"><option value="600+0">10 minutos</option><option value="600+5">10 min + 5 s</option><option value="900+10">15 min + 10 s</option><option value="300+0">5 minutos</option><option value="180+2">3 min + 2 s</option><option value="0+0">Sin reloj</option></select></label>
      <label>Respuesta de Jade<select id="pace"><option value="1">Natural</option><option value="1.5">Pausada</option><option value="0.5">Ágil</option></select></label>
    </div>
    <details><summary>Ajuste de estilo</summary><div class="advanced">
      <label>Influencia humana: <output id="weight-label">1.00</output><input id="weight" type="range" min="0" max="2" step=".25" value="1"></label>
      <div class="muted">1 es el valor inicial. 0 usa solo el motor y la temperatura. Los perfiles son orientativos, todavía no son ELO medidos.</div>
      <label style="margin-top:8px">Máximo de opciones<select id="maximum"><option>15</option><option>12</option></select></label>
      <div class="muted">Las opciones bajan hasta 3 con poco tiempo. Las jugadas humanas pueden entrar aunque no estén entre las favoritas del motor. El reloj es de práctica; puedes pausarlo.</div>
    </div></details>
    <button id="new" class="primary">Jugar</button>
  </details>
  <div class="confirm" id="confirm" hidden><b id="confirm-text"></b><div class="actions"><button id="confirm-yes">Confirmar</button><button id="confirm-no">Cancelar</button></div></div>
  <div class="status" role="status" aria-live="polite"><span id="turn">Listo para jugar</span><span id="message">Toca Jugar para empezar.</span></div>
  <div class="player"><div><b id="opponent">Jade · perfil 1300</b><div id="rhythm" class="muted">Rapid</div></div><span class="clock" id="clock-top" aria-label="Reloj superior">10:00</span></div>
  <div class="board-frame">
    <div class="ranks" id="ranks" aria-hidden="true"></div>
    <div class="board" id="board"><div id="drawing" aria-hidden="true"></div><div class="squares" id="squares" role="group" aria-label="Tablero de ajedrez"></div></div>
    <div class="files" id="files" aria-hidden="true"></div>
  </div>
  <div class="player"><div><b id="you">Tú · blancas</b><div class="muted">Toca origen y destino</div></div><span class="clock" id="clock-bottom" aria-label="Reloj inferior">10:00</span></div>
  <div class="promotion" id="promotion" role="group" aria-label="Promocionar peón" hidden><b>¿En qué pieza lo conviertes?</b><div class="promotions">
    <button data-piece="q">Dama</button><button data-piece="r">Torre</button><button data-piece="b">Alfil</button><button data-piece="n">Caballo</button>
  </div><button id="cancel-promotion">Cancelar</button></div>
  <button id="retry" class="primary" hidden>Reintentar respuesta de Jade</button>
  <div class="actions"><button id="undo">Deshacer turno</button><button id="flip">Girar tablero</button></div>
  <button id="pause" style="width:100%;margin-bottom:8px">Pausar reloj</button>
  <details id="history-panel"><summary>Movimientos <span id="count"></span></summary><div id="history" class="history"></div></details>
  <details><summary>Opciones de partida</summary>
    <div class="actions"><button id="draw">Reclamar tablas</button><button id="resign" class="danger">Rendirme</button></div>
    <label>Jugada por texto (UCI)<input id="uci" type="text" placeholder="e2e4" autocomplete="off" autocapitalize="none" spellcheck="false"></label>
    <button id="send" style="width:100%;margin-top:6px">Jugar movimiento</button>
    <div class="actions"><button id="download">Descargar PGN</button><button id="copy">Copiar PGN</button></div>
    <textarea id="pgn" readonly aria-label="Partida en PGN"></textarea>
    <div id="evidence" class="muted"></div>
  </details>
  <div class="footer" id="save-note"></div>
  <div class="footer">Piezas: C. M. Burnett · SVG de python-chess.<br>Las partidas jugadas no se añaden a la memoria humana.</div>
</main>
<script>
(()=>{
  const root=document.getElementById('jade');
  const el=id=>root.querySelector('#'+id);
  const callback='__JADE_CALLBACK__';
  let state=__JADE_INITIAL__, waiting=false, confirmation=null, clockStamp=performance.now(), flagSent=false;
  const status=text=>{el('message').textContent=text};
  function fit(){try{google.colab.output.setIframeHeight(Math.ceil(root.getBoundingClientRect().height)+4,true)}catch(e){}}
  function lock(){
    root.querySelectorAll('button,select,input').forEach(x=>x.disabled=waiting||state.closed);
    root.querySelectorAll('.squares button').forEach(x=>x.disabled=waiting||!state.can_move||state.promotion);
    el('undo').disabled=waiting||state.closed||!state.can_undo;
    el('draw').disabled=waiting||state.closed||!state.can_draw;
    el('resign').disabled=waiting||state.closed||!state.started||state.finished;
    el('send').disabled=waiting||!state.can_move||state.promotion;
    el('pause').disabled=waiting||state.closed||!state.started||state.finished;
    el('board').setAttribute('aria-busy',waiting?'true':'false');
  }
  function render(s){
    state=s;clockStamp=performance.now();flagSent=false;
    el('drawing').innerHTML=s.svg;
    el('ranks').replaceChildren(...s.ranks.map(x=>{const e=document.createElement('span');e.textContent=x;return e}));
    el('files').replaceChildren(...s.files.map(x=>{const e=document.createElement('span');e.textContent=x;return e}));
    el('squares').replaceChildren(...s.squares.map(q=>{
      const b=document.createElement('button');b.type='button';b.dataset.square=q.name;
      b.setAttribute('aria-label',q.label+(q.target?', destino legal':''));b.setAttribute('aria-pressed',q.selected?'true':'false');
      b.addEventListener('click',()=>send('square',{square:q.name}));return b;
    }));
    el('turn').textContent=s.closed?'Tablero cerrado':s.started?s.turn:'Listo para jugar';
    status(s.message);
    const humanLabel='Tú · '+(s.human?'blancas':'negras'), jadeLabel='Jade · perfil '+s.profile;
    el('opponent').textContent=s.orientation===s.human?jadeLabel:humanLabel;
    el('rhythm').textContent=s.rhythm==='rapid'?'Datos de rápidas':'Datos de blitz';
    el('you').textContent=s.orientation===s.human?humanLabel:jadeLabel;
    el('pause').textContent=s.clock.paused?'Continuar reloj':'Pausar reloj';
    el('new').textContent=s.started?'Nueva partida':'Jugar';
    el('retry').hidden=!s.ai_pending;
    el('promotion').hidden=!s.promotion;
    el('count').textContent=s.history.length?'('+s.history.length+')':'';
    const table=document.createElement('table');
    const header=document.createElement('tr');
    ['N.º','Blancas','Negras'].forEach(t=>{const th=document.createElement('th');th.textContent=t;header.append(th)});
    table.append(header);
    s.history.forEach(row=>{const tr=document.createElement('tr');row.forEach(t=>{const td=document.createElement('td');td.textContent=t;tr.append(td)});table.append(tr)});
    el('history').replaceChildren(table);el('history').scrollTop=el('history').scrollHeight;
    el('pgn').value=s.pgn;
    el('save-note').textContent=s.save_note;
    const d=s.decision;
    el('evidence').textContent=d?.candidate_count?
      d.candidate_count+' opciones · '+Math.round(100*d.human_mix)+' % de mezcla humana · '+d.candidate_observations+' ejemplos entre esas opciones. Última respuesta: '+(d.elapsed_seconds||0).toFixed(1)+' s.'+
      (d.selected_human_count?' La jugada elegida aparece '+d.selected_human_count+' veces en este perfil.':''):
      s.observations===null?'':s.observations+' observaciones humanas en la última respuesta.';
    lock();clocks();fit();
  }
  function clocks(){
    const c=state.clock;if(!c)return;
    const values=[...c.remaining];
    if(c.active!==null)values[c.active]=Math.max(0,values[c.active]-(performance.now()-clockStamp)/1000);
    const bottom=state.orientation?1:0;
    [['clock-top',1-bottom],['clock-bottom',bottom]].forEach(([id,side])=>{
      const t=values[side];el(id).textContent=c.enabled?(t<10?t.toFixed(1):Math.floor(t/60)+':'+String(Math.floor(t%60)).padStart(2,'0')):'Sin reloj';
      el(id).className='clock'+(c.active===side?' active':'')+(c.enabled&&t<=10?' low':'')+(c.paused?' paused':'');
    });
    if(c.active!==null&&values[c.active]<=0&&!waiting&&!flagSent&&!state.finished){flagSent=true;send('tick');}
  }
  async function call(action,payload={}){
    const result=await google.colab.kernel.invokeFunction(callback,[action,payload,state.revision],{});
    let data=result.data?.['application/json'];
    if(typeof data==='string')data=JSON.parse(data);
    if(!data||!data.squares)throw new Error('No se recibió el estado del tablero.');
    render(data);return data;
  }
  async function send(action,payload={}){
    if(waiting||state.closed)return;
    waiting=true;lock();
    try{
      if(action==='ai'){el('turn').textContent='Jade está pensando…';status('Calculando su respuesta.');el('retry').hidden=true;}
      const s=await call(action,payload);
      if(action==='new'){el('setup').open=false;fit()}
      if(s.ai_pending&&['new','square','promote','uci','pause'].includes(action)){
        el('turn').textContent='Jade está pensando…';status('Calculando su respuesta.');el('retry').hidden=true;
        // Permite pintar tu movimiento antes de iniciar la búsqueda del motor.
        await new Promise(resolve=>requestAnimationFrame(()=>requestAnimationFrame(resolve)));
        await call('ai');
      }
    }catch(err){
      status('No se pudo conectar. Comprueba «Conectar» en Colab y vuelve a abrir la celda 6. '+err.message);
    }finally{waiting=false;lock();fit()}
  }
  function confirm(text,action){confirmation=action;el('confirm-text').textContent=text;el('confirm').hidden=false;fit()}
  el('confirm-no').onclick=()=>{confirmation=null;el('confirm').hidden=true;fit()};
  el('confirm-yes').onclick=()=>{const fn=confirmation;confirmation=null;el('confirm').hidden=true;if(fn)fn()};
  el('new').onclick=()=>{
    const start=()=>send('new',{profile:Number(el('profile').value),color:el('color').value,beta:Number(el('weight').value),time_control:el('time-control').value,maximum:Number(el('maximum').value),pace:Number(el('pace').value)});
    if(state.plies&&!state.finished)confirm('¿Dejar esta partida y empezar otra?',start);else start();
  };
  el('weight').oninput=()=>el('weight-label').value=Number(el('weight').value).toFixed(2);
  el('undo').onclick=()=>send('undo');el('flip').onclick=()=>send('flip');el('draw').onclick=()=>send('draw');
  el('resign').onclick=()=>confirm('¿Quieres rendirte en esta partida?',()=>send('resign'));
  el('retry').onclick=()=>send('ai');
  el('pause').onclick=()=>send('pause');
  el('send').onclick=()=>{const uci=el('uci').value;el('uci').value='';send('uci',{uci})};
  el('uci').onkeydown=e=>{if(e.key==='Enter'){e.preventDefault();el('send').click()}};
  root.querySelectorAll('[data-piece]').forEach(b=>b.onclick=()=>send('promote',{piece:b.dataset.piece}));
  el('cancel-promotion').onclick=()=>send('cancel_promotion');
  el('download').onclick=()=>{
    const url=URL.createObjectURL(new Blob([state.pgn],{type:'application/x-chess-pgn'}));
    const a=document.createElement('a');a.href=url;a.download='Jade_partida.pgn';document.body.append(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),10000);
  };
  el('copy').onclick=async()=>{
    try{await navigator.clipboard.writeText(state.pgn);status('PGN copiado.');}
    catch(e){el('pgn').focus();el('pgn').select();status('Texto seleccionado. Mantén pulsado y elige Copiar.');}
  };
  root.querySelectorAll('details').forEach(d=>d.addEventListener('toggle',fit));
  el('profile').value=String(state.profile);el('color').value=state.human?'white':'black';
  el('weight').value=state.beta;el('weight-label').value=Number(state.beta).toFixed(2);
  el('maximum').value=state.maximum;el('pace').value=state.pace;
  el('time-control').value=state.clock.initial+'+'+state.clock.increment;
  el('setup').open=!state.started;render(state);
  const ticker=setInterval(()=>{if(!root.isConnected)clearInterval(ticker);else clocks()},200);
  if(typeof ResizeObserver!=='undefined')new ResizeObserver(fit).observe(root);
  window.addEventListener('resize',fit);
})();
</script>
'''

"""Reanálisis de PGN con Stockfish sin humanización. Exporta HTML, PGN y CSV."""
import csv
import html
import io
import json
import os
import time
import uuid
from pathlib import Path

import chess
import chess.engine
import chess.pgn
import chess.svg
from IPython.display import HTML, display


def jade_score_text(score):
    mate = score.mate()
    if mate is not None:
        return ("#" if mate >= 0 else "-#") + str(abs(mate))
    cp = score.score()
    return f"{cp / 100:+.2f}" if cp is not None else "?"


def jade_grade(best, played, same_move, ply):
    """Etiquetas propias, basadas en pérdida de expectativa WDL, no de Lichess."""
    best_mate, played_mate = best.mate(), played.mate()
    loss_cp = None if best.is_mate() or played.is_mate() else max(0, best.score() - played.score())
    loss_wdl = max(0., best.wdl(model="sf", ply=ply).expectation() - played.wdl(model="sf", ply=ply).expectation())
    if same_move:
        return "Mejor del análisis", 0 if loss_cp is not None else None, 0.
    if best_mate is not None and best_mate > 0 and (played_mate is None or played_mate <= 0):
        return "Mate omitido", loss_cp, loss_wdl
    if played_mate is not None and played_mate <= 0 and (best_mate is None or best_mate > 0):
        return "Permite mate", loss_cp, loss_wdl
    if best_mate is not None and played_mate is not None:
        return "Mantiene mate" if played_mate > 0 else "Posición con mate en contra", loss_cp, loss_wdl
    label = ("Precisa" if loss_wdl < .02 else "Buena" if loss_wdl < .05 else
             "Imprecisión" if loss_wdl < .10 else "Error" if loss_wdl < .20 else "Error grave")
    return label, loss_cp, loss_wdl


def jade_report_html(rows, metadata):
    """Informe autónomo. El tablero de cada jugada se prepara antes de exportar."""
    payload = json.dumps(dict(rows=rows, metadata=metadata), ensure_ascii=False).replace("<", "\\u003c")
    return '''<!doctype html><html lang="es"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Jade · Análisis de partida</title><style>
*{box-sizing:border-box}body{margin:0;background:#f5f6ef;color:#173c30;font:15px/1.5 system-ui,sans-serif}
main{max-width:1050px;margin:auto;padding:16px}h1{margin:0 0 8px}p{margin:8px 0}.layout{display:grid;grid-template-columns:minmax(220px,400px) minmax(0,1fr);gap:20px}.layout>section{min-width:0}
#board svg{width:100%;height:auto;display:block}button,select{font:inherit;border:1px solid #bbccbf;border-radius:8px;background:white;color:#173c30;padding:10px;cursor:pointer;min-height:44px}
.controls{display:flex;gap:6px;flex-wrap:wrap;margin:12px 0}.scroll{overflow:auto;max-height:540px;background:white;border-radius:10px}table{width:100%;border-collapse:collapse;font-size:13px;white-space:nowrap}
th,td{text-align:left;padding:9px;border-bottom:1px solid #e2e9db}tr[data-i]{cursor:pointer}tr.selected{background:#dceac8}tr:hover{background:#edf2e3}.note{color:#4f6759;font-size:13px}
#explain{padding:12px;border-radius:10px;background:#e5eddf;min-height:110px;overflow-wrap:anywhere}.bad{color:#a0392f}@media(max-width:690px){.layout{grid-template-columns:minmax(0,1fr)}main{padding:12px}.scroll{max-height:340px}}
</style><main><h1>Jade · Análisis con Stockfish</h1><p id="game"></p><p class="note" id="meta"></p><p class="bad" id="warning"></p>
<p class="note">Evaluaciones desde el punto de vista de las blancas: positivo favorece a blancas, negativo a negras. Las etiquetas son heurísticas propias. WDL estima resultados del motor, no tus probabilidades reales de ganar.</p>
<div class="layout"><section><div id="board"></div><div class="controls"><button id="prev">Anterior</button><button id="next">Siguiente</button><button id="line">Ver mejor continuación</button><button id="back">Posición inicial</button></div><div id="explain"></div></section>
<section><div class="controls"><label>Ver <select id="side"><option value="">Ambos colores</option><option value="Blancas">Blancas</option><option value="Negras">Negras</option></select></label></div><div class="scroll"><table><thead><tr><th>Jugada</th><th>Valoración</th><th>Mejor</th><th>Eval.</th><th>Pérdida cp</th></tr></thead><tbody id="rows"></tbody></table></div></section></div></main>
<script>(()=>{const d=__DATA__,e=id=>document.getElementById(id);let index=0,line=false;
e('game').textContent=d.metadata.white+' contra '+d.metadata.black+' · '+d.metadata.result;
e('meta').textContent=d.metadata.engine+' · '+d.metadata.seconds+' s por búsqueda · '+d.metadata.threads+' hilos · '+d.rows.length+'/'+d.metadata.total+' jugadas analizadas.';
e('warning').textContent=d.metadata.complete?'':'ANÁLISIS INCOMPLETO. '+(d.metadata.error||'Se interrumpió la ejecución.');
function show(i){if(!d.rows.length){e('explain').textContent='No hay jugadas analizadas.';return}index=Math.max(0,Math.min(d.rows.length-1,i));line=false;const r=d.rows[index];e('board').innerHTML=r.svg;
e('explain').replaceChildren();const a=document.createElement('b');a.textContent=r.turn+' '+r.played+' · '+r.label;const b=document.createElement('p');b.textContent='Stockfish prefiere '+r.best+'. Continuación: '+r.pv;const c=document.createElement('p');c.textContent='Evaluación de tu jugada: '+r.evaluation+'; profundidad '+r.depth+'.'+(r.loss_cp===null?' Mate: no se convierte a centipeones ficticios.':' Pérdida estimada: '+r.loss_cp+' cp.');e('explain').append(a,b,c);
e('rows').querySelectorAll('tr').forEach(tr=>tr.classList.toggle('selected',Number(tr.dataset.i)===index));}
function table(){e('rows').replaceChildren();d.rows.forEach((r,i)=>{if(e('side').value&&r.color!==e('side').value)return;const tr=document.createElement('tr');tr.dataset.i=i;[r.turn+' '+r.played,r.label,r.best,r.evaluation,r.loss_cp===null?'mate':r.loss_cp].forEach(t=>{const td=document.createElement('td');td.textContent=t;tr.append(td)});tr.onclick=()=>show(i);e('rows').append(tr)});show(index)}
e('side').onchange=table;e('prev').onclick=()=>show(index-1);e('next').onclick=()=>show(index+1);e('back').onclick=()=>show(index);e('line').onclick=()=>{if(!d.rows.length)return;line=!line;e('board').innerHTML=line?d.rows[index].pv_svg:d.rows[index].svg};table();})();</script></html>'''.replace("__DATA__", payload)


def analizar_pgn_jade(text, binary, output_dir, seconds=3., threads=None, hash_mb=512, log=print):
    """Analiza una partida con un proceso independiente a máxima fuerza configurada.

    Cada jugada compara búsqueda libre y búsqueda forzando la jugada real, con
    el mismo tiempo por búsqueda. No usa la temperatura, el reloj ni los datos
    humanos de Jade. El tiempo es finito: ninguna profundidad garantiza perfección.
    """
    seconds = float(seconds)
    if not .01 <= seconds <= 120:
        raise ValueError("Elige entre 0,01 y 120 segundos por búsqueda.")
    stream = io.StringIO(str(text).lstrip("\ufeff"))
    game = chess.pgn.read_game(stream)
    if game is None or game.errors:
        raise ValueError("No se pudo leer el PGN completo. Revisa las jugadas y la posición inicial.")
    if chess.pgn.read_game(stream) is not None:
        raise ValueError("El archivo contiene varias partidas. Exporta una sola para este analizador.")
    board = game.board()
    if not board.is_valid() or board.chess960 or game.headers.get("Variant", "Standard") not in ("Standard", "Chess"):
        raise ValueError("Este analizador admite ajedrez estándar con posición inicial válida.")
    moves = list(game.mainline_moves())
    if not moves:
        raise ValueError("El PGN no contiene movimientos.")
    thread_count = max(1, min(int(threads or min(os.cpu_count() or 1, 4)), os.cpu_count() or 1))
    hash_mb = max(16, min(int(hash_mb), 2048))
    folder = Path(output_dir) / ("analisis_" + time.strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6])
    folder.mkdir(parents=True, exist_ok=False)
    annotated = chess.pgn.Game()
    annotated.headers.update(game.headers)
    annotated.setup(game.board())
    annotated.headers["Annotator"] = "Jade · Stockfish sin humanización"
    node, rows = annotated, []
    metadata = dict(white=game.headers.get("White", "Blancas"), black=game.headers.get("Black", "Negras"),
                    result=game.headers.get("Result", "*"), total=len(moves), seconds=seconds,
                    threads=thread_count, complete=False, error="", engine="Stockfish")

    def export():
        metadata["complete"] = len(rows) == len(moves)
        annotated.headers["Result"] = metadata["result"] if metadata["complete"] else "*"
        (folder / "partida_analizada.pgn").write_text(str(annotated), encoding="utf-8")
        fields = ["turn", "color", "played", "best", "evaluation", "loss_cp", "loss_wdl", "label", "depth", "pv"]
        with (folder / "movimientos.csv").open("w", encoding="utf-8-sig", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=fields, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(rows)
        (folder / "informe.html").write_text(jade_report_html(rows, metadata), encoding="utf-8")

    engine = None
    try:
        engine = chess.engine.SimpleEngine.popen_uci(str(binary), timeout=seconds + 60)
        engine.configure({"Threads": thread_count, "Hash": hash_mb, "Skill Level": 20,
                          "UCI_LimitStrength": False})
        metadata["engine"] = engine.id.get("name", "Stockfish")
        token = object()
        log(f"{metadata['engine']} a máxima fuerza configurada. Hasta ~{len(moves) * 2 * seconds / 60:.1f} min de búsqueda, más exportación.")
        original_nodes = list(game.mainline())
        for index, move in enumerate(moves):
            mover = board.turn
            limit = chess.engine.Limit(time=seconds)
            best_info = engine.analyse(board, limit, multipv=1, game=token)[0]
            best_move = best_info["pv"][0]
            played_info = best_info if move == best_move else engine.analyse(board, limit, root_moves=[move], game=token)
            best_score = best_info["score"].pov(mover)
            played_score = played_info["score"].pov(mover)
            label, cp_loss, wdl_loss = jade_grade(best_score, played_score, move == best_move, board.ply())
            if board.legal_moves.count() == 1:
                label = "Única jugada legal"
            pv = best_info["pv"][:8]
            continuation = board.variation_san(pv)
            future = board.copy(stack=True)
            for next_move in pv:
                future.push(next_move)
            san, best_san = board.san(move), board.san(best_move)
            turn = f"{board.fullmove_number}." if mover else f"{board.fullmove_number}..."
            arrows = [chess.svg.Arrow(best_move.from_square, best_move.to_square, color="#21885ac0")]
            if move != best_move:
                arrows.append(chess.svg.Arrow(move.from_square, move.to_square, color="#c8503ab0"))
            row = dict(turn=turn, color="Blancas" if mover else "Negras", played=san, best=best_san,
                       evaluation=jade_score_text(played_info["score"].white()),
                       loss_cp=cp_loss, loss_wdl=round(wdl_loss, 4), label=label,
                       depth=played_info.get("depth", 0), pv=continuation,
                       svg=chess.svg.board(board, arrows=arrows), pv_svg=chess.svg.board(future, lastmove=pv[-1]))
            rows.append(row)
            node = node.add_variation(move)
            node.comment = f"Jade: {label}. Mejor: {best_san}. Línea: {continuation}."
            node.set_eval(played_info["score"], played_info.get("depth"))
            original_clock = original_nodes[index].clock()
            if original_clock is not None:
                node.set_clock(original_clock)
            if move != best_move:
                alternative = node.parent.add_line(pv)
                alternative.comment = "Continuación preferida por Stockfish."
            board.push(move)
            log(f"{index+1}/{len(moves)} · {turn} {san}: {label}; mejor {best_san}")
            if (index + 1) % 5 == 0:
                export()
    except KeyboardInterrupt:
        metadata["error"] = "Interrumpido por el usuario; se conserva el análisis completado."
        log(metadata["error"])
    except Exception as exc:
        metadata["error"] = str(exc)
        log("Análisis incompleto: " + str(exc))
    finally:
        if engine is not None:
            try:
                engine.quit()
            except chess.engine.EngineTerminatedError:
                pass
        export()
    return dict(rows=rows, metadata=metadata, directory=folder,
                html=folder / "informe.html", pgn=folder / "partida_analizada.pgn", csv=folder / "movimientos.csv")


def abrir_analizador_jade(jade, tablero=None, output_dir=None):
    """Formulario para pegar PGN, subir un archivo o cargar la partida de Jade."""
    import ipywidgets as widgets
    if output_dir is None:
        game_path = getattr(tablero, "game_path", None)
        output_dir = (game_path.parent.parent if game_path else Path("/content/Jade")) / "analisis"
    text = widgets.Textarea(placeholder="Pega aquí el PGN completo de Lichess o Chess.com",
                           layout=widgets.Layout(width="100%", height="160px"))
    upload = widgets.FileUpload(accept=".pgn,.txt", multiple=False, description="Subir PGN")
    current = widgets.Button(description="Usar partida de Jade", layout=widgets.Layout(width="100%"))
    seconds = widgets.Dropdown(options=[("Rápido: 1 s", 1), ("Normal: 3 s", 3), ("Profundo: 10 s", 10), ("Muy profundo: 30 s", 30)], value=3,
                               description="Análisis:", layout=widgets.Layout(width="100%"))
    run = widgets.Button(description="Analizar con Stockfish", button_style="success", layout=widgets.Layout(width="100%"))
    out = widgets.Output(layout=widgets.Layout(width="100%"))

    def use_current(_):
        if tablero is not None:
            if hasattr(tablero, "settle_clock"):
                tablero.settle_clock()
                tablero.paused, tablero.clock_started = True, None
                tablero.save_game()
            text.value = tablero.pgn()

    def uploaded(change):
        items = change["new"]
        if items:
            first = next(iter(items.values())) if isinstance(items, dict) else items[0]
            try:
                text.value = bytes(first["content"]).decode("utf-8-sig")
            except UnicodeDecodeError:
                text.value = bytes(first["content"]).decode("latin-1")

    def analyse(_):
        run.disabled = True
        with out:
            out.clear_output()
            try:
                if tablero is not None and hasattr(tablero, "settle_clock"):
                    tablero.settle_clock()
                    tablero.paused, tablero.clock_started = True, None
                    tablero.save_game()
                result = analizar_pgn_jade(text.value, jade.binary, output_dir, seconds=seconds.value)
                print("Archivos guardados en:", result["directory"])
                display(HTML(result["html"].read_text(encoding="utf-8")))
                # Descargas gestionadas por Colab, disponibles también para móvil.
                for name in ("html", "pgn", "csv"):
                    button = widgets.Button(description="Descargar " + name.upper())
                    def download(_, path=result[name]):
                        from google.colab import files
                        files.download(str(path))
                    button.on_click(download)
                    display(button)
            except Exception as exc:
                print("No se pudo analizar:", exc)
            finally:
                run.disabled = False

    current.on_click(use_current)
    upload.observe(uploaded, names="value")
    run.on_click(analyse)
    root = widgets.VBox([widgets.HTML("<b>Analizar PGN con Stockfish</b><br>Pega el texto, sube un archivo o usa tu partida. Se pausará el reloj de práctica. Las etiquetas son orientativas."),
                         current, upload, text, seconds, run, out],
                        layout=widgets.Layout(width="100%", max_width="800px", min_width="0"))
    display(root)
    return root


def memory_report(memory):
    summary = memory.summary()
    print(f"Jade · {memory.rhythm}: {summary['partidas']:,} partidas únicas; base local {summary['base_MB']} MB")
    print("Perfil | Partidas con ese perfil | Movimientos")
    for profile in PROFILES:
        values = summary["perfiles"].get(profile, {"partidas": 0, "movimientos": 0})
        print(f"{profile:6} | {values['partidas']:23,} | {values['movimientos']:,}")
    return summary


## 4 · Preparar Jade y recuperar su memoria
En una sesión nueva se recupera la última copia válida. En la sesión actual se
conserva la base local, que puede estar más actualizada. Los IDs de partidas,
frecuencias, ritmo, revisión del dataset y cursores se guardan juntos.


In [ ]:
#@title 4. Iniciar o recuperar Jade
if "JadeMemory" not in globals() or "DB_PATH" not in globals():
    raise RuntimeError("Falta preparar esta sesión. Ejecuta las celdas 1, 2 y 3, en orden.")
if "tablero_jade" in globals():
    tablero_jade.close()
if "jade" in globals():
    jade.close()
if "memoria" in globals():
    memoria.close()

restore_memory(DB_PATH, BACKUP_DIR, RITMO)
memoria = JadeMemory(DB_PATH, RITMO)
binary = shutil.which("stockfish") or "/usr/games/stockfish"
jade = JadeEngine(binary, memoria, nodes=NODOS_STOCKFISH)
print("Motor:", jade.engine.id.get("name"), "| Nodos:", NODOS_STOCKFISH)
with memoria.conn:
    memoria.set_meta("last_engine", {"name": jade.engine.id.get("name"), "nodes": NODOS_STOCKFISH})
memory_report(memoria)


## 5 · Alimentar a Jade con otro lote
No necesitas descargar archivos. Se consultan las columnas requeridas del Parquet
remoto. Cada ejecución **añade** partidas nuevas y continúa donde terminó la anterior.
Filtra el ritmo, descarta cuentas identificadas como BOT y usa el rating de quien
mueve. Recoge los primeros 100 movimientos individuales por partida (50 turnos).

Los límites son por ejecución y para el conjunto de perfiles, no por perfil.
Si llega antes a `MAX_FILAS` o `MAX_ARCHIVOS`, puede añadir menos partidas.
El muestreo es exploratorio: los archivos se visitan en un orden mezclado fijo,
pero no garantizamos que la muestra represente a todo Lichess.

Se guarda al terminar y, durante cargas largas, en puntos de control aproximadamente
cada 2 minutos (entre lotes). Si Colab se desconecta de golpe, puede perderse lo
posterior a la última copia completa; al reanudar no se duplican las partidas guardadas.
Si detienes con el botón de parar, el código intenta guardar lo confirmado.

Parquet permite leer rangos de bytes, pero **no equivale a descargar solamente las
partidas del ELO elegido**: primero hay que leer bloques y luego filtrarlos. Reanudar
dentro de un bloque puede volver a transferir parte de ese bloque.


In [ ]:
#@title 5. Añadir partidas nuevas y guardar
if "memoria" not in globals():
    raise RuntimeError("La memoria no está abierta. Ejecuta primero las celdas 1, 2, 3 y 4.")
if memoria.rhythm != RITMO:
    raise ValueError("Has cambiado de ritmo: ejecuta primero la celda 4.")

resultado_carga = feed_jade(
    memoria,
    year=ANO,
    month=MES,
    new_games=NUEVAS_PARTIDAS,
    max_rows=MAX_FILAS,
    max_files=MAX_ARCHIVOS,
    backup_dir=BACKUP_DIR,
)
print("Resultado de este lote:", resultado_carga)
memory_report(memoria)


## 6 · Jugar contra Jade
Elige perfil, blancas/negras y pulsa **Nueva partida**. Toca la pieza de origen y
la casilla de destino; los puntos y los círculos señalan destinos legales. También puedes
escribir UCI desde «Opciones de partida». Para enrocar, toca el rey y su destino.
Al promocionar aparece un selector de pieza. Las letras del historial siguen la notación PGN inglesa.

El dibujo completo utiliza `chess.svg.board` de python-chess y piezas vectoriales
de C. M. Burnett. No depende de fuentes de símbolos ni descarga piezas de otro servidor.
El tablero se ajusta al ancho de la salida de Colab. Si el navegador sigue mostrando
un desplazamiento horizontal antiguo, vuelve al extremo izquierdo y recarga la salida.

El botón Deshacer turno permite practicar de nuevo. También puedes rendirte,
reclamar tablas o copiar el PGN desde el desplegable. Los ajustes se aplican a una
nueva partida. El reloj inicial es **10 minutos**, sin incremento. También puedes elegir
10+5, 15+10, 5+0, 3+2 o sin reloj. `rapid/blitz` sigue identificando la memoria humana:
cambiar el reloj en el tablero no cambia ni mezcla esas bases de datos.

El reloj descuenta el tiempo real de cada turno, incluido cálculo y pausa de Jade.
El incremento se suma tras mover. Puedes pausarlo: es un reloj de práctica.
Después de reiniciar o volver a abrir el tablero, la partida se recupera **en pausa**
con los tiempos guardados; no se penaliza el tiempo desconectado. Una desconexión abrupta
puede perder el tiempo transcurrido desde el último guardado. Deshacer recupera los
tiempos guardados antes de las jugadas retiradas. No es un reloj homologado para competición.
Desde esta versión, la partida en curso se guarda después de cada movimiento en
`Mi unidad/Jade/partidas/actual_RITMO.json` si Drive está activado. Volver a ejecutar
6 conserva la partida; después de un reinicio ejecuta 1 a 4 y 6 para recuperarla.
Si te desconectas entre tu jugada y la respuesta, pulsa «Continuar reloj»; si hace falta,
pulsa «Reintentar respuesta de Jade».
Al iniciar otra partida, la anterior se archiva como PGN. Puedes descargar su PGN desde
«Opciones de partida» en cualquier momento. Esto no entrena a Jade.
Las partidas contra Jade **no se añaden automáticamente a su memoria**, para no
mezclar respuestas del motor con las partidas humanas del dataset.

Peso humano 0 = solo Stockfish + temperatura; 1 = valor inicial. Más peso da más
influencia a las frecuencias observadas, sin garantizar mayor realismo ni menor fuerza.
Si aparece «0 observaciones», Jade usa Boltzmann para esa respuesta: no inventa datos.


In [ ]:
#@title 6. Abrir el tablero
if "jade" not in globals() or "memoria" not in globals():
    raise RuntimeError("Primero ejecuta las celdas 1, 2, 3 y 4 para recuperar la memoria.")

anterior = globals().get("tablero_jade")
if anterior is not None and anterior.jade.memory.rhythm != jade.memory.rhythm:
    anterior.close()
    anterior = None
carpeta_partidas = (Path(BACKUP_DIR).parent if BACKUP_DIR is not None else Path(DB_PATH).parent) / "partidas"
if anterior is not None:
    anterior.close()
nuevo_tablero = JadeBoard(jade, game_path=carpeta_partidas / f"actual_{jade.memory.rhythm}.json", previous=anterior)
tablero_jade = nuevo_tablero
tablero_jade.show()


## 7 · Guardar cuando quieras
La alimentación ya guarda automáticamente. Esta celda permite guardar otra copia
manualmente. Se conservan las tres más recientes por ritmo, para recuperar la anterior
si la última estuviese dañada. Las frecuencias crecen en disco, no en un diccionario
gigante de RAM. El tamaño exacto depende del número de posiciones diferentes.


In [ ]:
#@title 7. Guardar memoria manualmente
ultima_copia = save_memory(memoria, BACKUP_DIR)
memory_report(memoria)
if BACKUP_DIR is None:
    print("Sin Drive: descarga esta copia desde el panel Archivos de Colab:", ultima_copia)


## 8 · Analizar una partida PGN con Stockfish

Ejecuta esta celda y pega el PGN completo, pulsa «Subir PGN» o «Usar partida de Jade».
Elige 1, 3, 10 o 30 segundos **por búsqueda** y pulsa «Analizar con Stockfish».
El análisis usa un proceso independiente, `Skill Level=20`, sin límite de fuerza,
sin aleatoriedad, hasta 4 hilos disponibles y 512 MB de hash. No utiliza los límites
de nodos, el reloj ni la selección humana del rival.

Cada jugada requiere una búsqueda libre y, si difiere de la mejor, otra forzando esa
jugada. Una partida de 80 movimientos individuales a 3 s puede necesitar aproximadamente
8 minutos de cálculo, más la exportación. Más tiempo puede mejorar el análisis; no existe
un ajuste finito que garantice encontrar siempre la jugada óptima. La profundidad real
alcanzada se muestra en cada fila. Los mates se tratan como mates, no como cp ficticios.

El informe muestra **evaluaciones desde blancas**, una etiqueta orientativa por jugada,
la mejor alternativa y su continuación. Las etiquetas utilizan pérdida de expectativa
WDL del motor; no reproducen las etiquetas ni la precisión de Lichess/Chess.com, ni
representan probabilidades de victoria de jugadores humanos. La pérdida cp se compara
desde el bando que mueve. Puede variar al aumentar el tiempo de búsqueda.

Exporta HTML interactivo, PGN anotado y CSV a `Mi unidad/Jade/analisis` con Drive activado.
El HTML funciona sin Colab una vez descargado. Las flechas verdes indican la alternativa
del motor y las rojas la jugada real si es diferente. Puedes filtrar blancas/negras.
Se guardan avances cada cinco jugadas y al finalizar. Un análisis interrumpido queda
marcado como incompleto. Se admite una partida estándar por archivo, con su FEN si lo trae.
Al analizar se pausa la partida de práctica; vuelve a ejecutar 6 para sincronizar el tablero.


In [ ]:
#@title 8. Abrir el analizador PGN
if "jade" not in globals() or "abrir_analizador_jade" not in globals():
    raise RuntimeError("Ejecuta primero las celdas 1 a 4 de Jade 1.2.")
carpeta_analisis = (Path(BACKUP_DIR).parent if BACKUP_DIR is not None else Path(DB_PATH).parent) / "analisis"
panel_analisis = abrir_analizador_jade(jade, globals().get("tablero_jade"), carpeta_analisis)


## Añadir más datos sin empezar de cero

- **Otro lote del mismo mes:** vuelve a ejecutar 5. Continúa desde el cursor guardado.
- **Lote mayor:** cambia `NUEVAS_PARTIDAS` a 5000 en 2, ejecuta 2 y después 5.
  Si faltan partidas aceptadas, aumenta `MAX_FILAS` (por ejemplo a 100000).
- **Otro mes:** cambia `MES` en 2, ejecuta 2 y después 5. Se añade a la misma memoria.
  Al volver al mes anterior recuerda su cursor. Mantén el mismo ritmo.
- **Otro ritmo:** cambia `RITMO`, ejecuta 2, 4, 5 y 6. Se usa otra base para no mezclar ratings.
- **Volver mañana:** ejecuta 1, 2, 3, 4 y 6. La carga 5 es opcional.
- **Error de red o límite de Hugging Face:** espera lo que indique el servicio y vuelve
  a ejecutar 5. No borres la memoria. No necesitas un token para el dataset público;
  si el servicio exige autenticación o limita el acceso, usa su inicio de sesión oficial.
- **Mes sin archivos:** el dataset queda fijado a una revisión. Escoge un mes existente;
  los meses añadidos después no aparecen automáticamente en esa revisión.
- **Drive lleno o error de copia:** la base local sigue en `/content/Jade`.
  Resuelve el problema de Drive y ejecuta 7 antes de cerrar Colab.
- **Tablero sin responder:** comprueba que Colab sigue conectado; ejecuta 6 de nuevo.
  Si el motor se cerró, ejecuta 4 y 6. La nueva interfaz recupera la partida guardada.

Guardar el `.ipynb` no guarda por sí mismo la memoria. La carpeta de copias sí la guarda.
El tablero tampoco es un servicio permanente: necesita que el entorno de Colab siga vivo.


## Archivos: qué hace falta y dónde va

| Elemento | Procedencia | Destino | ¿Lo subes a mano? |
|---|---|---|---|
| `Jade_Colab.ipynb` | Este cuaderno | Colab / tu Drive | Solo la primera vez |
| Binario Stockfish y NNUE | Paquete Ubuntu | Entorno temporal de Colab | No |
| Bibliotecas Python | PyPI | Entorno temporal de Colab | No |
| Parquet de partidas | Hugging Face, Lichess | Lectura remota por bloques | No |
| `jade_rapid.sqlite` o `jade_blitz.sqlite` | Lo crea Jade | `/content/Jade` | No |
| `jade_rapid_FECHA.sqlite.gz` | Copia comprimida | `Mi unidad/Jade/copias` | No |
| `actual_rapid.json` / `actual_blitz.json` | Partida en curso | `Mi unidad/Jade/partidas` | No |
| `partida_ID.pgn` | Partida anterior al pulsar Nueva partida | `Mi unidad/Jade/partidas` | No |

No hacen falta `chess-evaluations`, tablas de finales Syzygy, pesos propios ni una
copia completa de la base de Lichess. Las evaluaciones las calcula Stockfish al jugar.
Un PGN personal tampoco es necesario para alimentar esta versión.

Comprobación de tamaño: enero de 2024, en la revisión inspeccionada del dataset,
contiene **421 archivos, unos 71,73 GB decimales**. El primero ocupa unos 168,17 MB
y contiene 235.142 partidas, con bloques de 1.000 filas salvo el último. Estos datos
describen ese mes y archivo concretos, no garantizan tamaños iguales para otros.


## Qué aprende y qué no podemos prometer

Jade guarda cuántas veces se jugó cada movimiento desde una posición, separando
bandas de rating de 100 puntos. Para 1300 usa 1250–1349. No reentrena Stockfish.

| Tiempo que le queda a Jade | Máximo de opciones consideradas |
|---|---|
| Más de 180 s o sin reloj | 15 (o 12, según tu ajuste) |
| Más de 90 y hasta 180 s | 12 |
| Más de 30 y hasta 90 s | 8 |
| Más de 10 y hasta 30 s | 5 |
| Hasta 10 s | 3 |

Si hay menos jugadas legales, se consideran las disponibles. No se inventan tres
jugadas cuando solo hay una. Un incremento de tiempo puede volver a aumentar el número.

La lista final mezcla candidatas del motor y jugadas legales observadas **al menos dos
veces en ese perfil**, aunque no estén en el ranking inicial de Stockfish. En las primeras
24 medias jugadas (12 turnos) se reservan hasta dos tercios de las plazas para esas
jugadas humanas; después, hasta la mitad. Siempre se conserva la mejor referencia
inicial del motor. El máximo se aplica a la lista final, no a los nodos internos.

Una comprobación táctica breve compara las candidatas. La distribución final es una
mezcla: `p = (1 - mezcla_humana) * softmax_motor + mezcla_humana * frecuencia_ajustada`.
La mezcla humana aumenta con la evidencia y tiene más peso en la apertura; la frecuencia
se penaliza si la pérdida estimada supera un margen. Esto admite gambitos comunes y
movimientos subóptimos. Evita mates en contra detectados si hay alternativa, pero una
búsqueda corta puede pasar por alto tácticas. No garantiza jugar como un humano real.

Con poco tiempo se reduce también la búsqueda y aumenta la temperatura, para no volver
más preciso al rival simplemente al reducir las opciones. La respuesta en modo Natural
suele rondar 2–9 s con tiempo abundante, y décimas con apuros. La pausa se adapta a la
ambigüedad y a la familiaridad de la apertura; es una heurística, no tiempos aprendidos
de los relojes de Lichess. El tiempo de CPU/red/Drive puede alargar la respuesta.

La temperatura, la mezcla y las pausas son heurísticas aún sin calibrar. Más datos dan
más cobertura, no un ELO medido. En posiciones nuevas sigue predominando Stockfish.
La clave de frecuencias agrupa transposiciones y omite contadores e historial; el motor
recibe el historial real. Cambiar la interfaz o analizar PGN no añade observaciones humanas.

No se muestran porcentajes de «jugadores que encontraron una jugada»: los contadores
son **ocurrencias de movimientos**, no personas distintas. La interfaz muestra el número
de observaciones que realmente respaldaron la última elección.


## Fuentes verificadas

- [Lichess Standard Chess Games, formato Parquet y licencia CC0](https://huggingface.co/datasets/Lichess/standard-chess-games)
- [Ficha: particiones por año y mes, PGN en movetext y ratings Glicko-2](https://huggingface.co/datasets/Lichess/standard-chess-games/blob/main/README.md)
- [Base original de Lichess, descargas PGN comprimidas](https://database.lichess.org/)
- [Hugging Face: acceso a archivos remotos con HfFileSystem](https://huggingface.co/docs/huggingface_hub/guides/hf_file_system)
- [Arrow: lectura de columnas, bloques y lotes de Parquet](https://arrow.apache.org/docs/python/generated/pyarrow.parquet.ParquetFile.html)
- [python-chess: motor UCI y MultiPV](https://python-chess.readthedocs.io/en/latest/engine.html)
- [Stockfish utiliza CPU](https://official-stockfish.github.io/docs/stockfish-wiki/Stockfish-FAQ.html#can-stockfish-use-my-gpu)
- [Colab: sesiones temporales, Drive y reducción de operaciones pequeñas](https://research.google.com/colaboratory/faq.html)
- [Tablero y piezas SVG de python-chess](https://python-chess.readthedocs.io/en/latest/svg.html)
- [Comunicación de JavaScript con Python dentro de Colab](https://colab.research.google.com/notebooks/snippets/advanced_outputs.ipynb)

El código se verifica con Stockfish y un lote real fuera de Colab. El montaje de
tu Drive requiere tu cuenta y solo se realiza cuando tú ejecutes el cuaderno.

La interfaz se verifica en Chromium a tamaños de móvil y escritorio, con las
llamadas al kernel enlazadas a Python en el entorno de pruebas. Se verifican movimientos
legales, ambos colores, giro, enroque, promoción, deshacer, PGN y recuperación de partida.
La conexión autenticada de Colab, tu Drive y el modo oscuro forzado de Opera requieren
comprobación en tu dispositivo.
